In [1]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

#paths_bonner = DatasetBonnerPNAS2017.load_dataset()
#pprint(paths_bonner)

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

# count files
# num_images = len([f for f in os.listdir(stimuli_path) if f.endswith(('.jpg', '.png'))])

#print("Number of images:", num_images)

In [4]:
from torchvision import transforms as T
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor

def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = transforms(image).unsqueeze(0)
    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor


def my_extactor(preprocessed_data, layers_to_extract, model):
    # Create a extractor instance
    extractor_model = tx.Extractor(model, layers_to_extract)
    
    # Extract actual features
    _, features = extractor_model(preprocessed_data)

    return features

#def my_extactor_jepa(preprocessed_data, layers_to_extract, model):
    #features = model(preprocessed_data)
    #return {"jepa": features}
    
def my_transformer_extractor(preprocessed_data, layers_to_extract, model):
    # call the ORIGINAL ViT model (not the wrapper) and returns features of every layer
    
    outputs = model.model(preprocessed_data, output_hidden_states=True)  # outputs all layers too. You completely bypass JEPAFeatureExtractor.forward()
    # if it was model() then it uses JEPAFeatureExtractor.
    hidden_states = outputs.hidden_states

    features = {}
    for i, layer in enumerate(hidden_states):
        layer = layer[:, 1:, :]      # remove CLS
        layer = layer.mean(dim=1)    # pool
        features[f"layer_{i}"] = layer

    return features
    
def my_laste_layer_transformer_extractor(preprocessed_data, layers_to_extract, model):
    # call the ORIGINAL ViT model (not the wrapper) 
    outputs = model.model(preprocessed_data)  

    tokens = outputs.last_hidden_state
    tokens = tokens[:, 1:, :]
    features = tokens.mean(dim=1)

    return features

        
def my_extactor_lejepa(preprocessed_data, layers_to_extract, model):
    
    features = {}

    # Patch embedding
    x = model.model.patch_embed(preprocessed_data)

    # Add CLS token
    cls_token = model.model.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)

    # Add positional embeddings
    x = x + model.model.pos_embed
    x = model.model.pos_drop(x)

    # Number of transformer blocks automatically detected
    num_blocks = len(model.model.blocks)

    # Extract output of every transformer block
    for i, block in enumerate(model.model.blocks):
        x = block(x)

        # For the last block, keep BOTH versions:
        if i == num_blocks - 1:
            features[f"layer_{i}_pre_norm"] = x[:, 0, :]

        else:
            features[f"layer_{i}"] = x[:, 0, :]

    # Final LayerNorm. last transformer-block output before LayerNorm showd worse results than final output after LayerNorm
    x = model.model.norm(x)
    features[f"layer_{num_blocks - 1}_post_norm"] = x[:, 0, :]

    return features
    
def my_cleaner(features):
    return features



In [5]:
import torch.nn as nn

class JEPAFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model #You completely bypass JEPAFeatureExtractor.forward()

    def forward(self, x):
        with torch.no_grad():
            outputs = self.model(x) #Run the actual transformer. 

        tokens = outputs.last_hidden_state  # (B, 257, 1280). Get all patch representations

        # remove fake CLS token
        tokens = tokens[:, 1:, :]

        # average pooling
        features = tokens.mean(dim=1)  # (B, 1280)

        return features

jepa_extractor = JEPAFeatureExtractor(vit_model)

NameError: name 'vit_model' is not defined

# IJEPA

In [ ]:
# load JEPA model
from safetensors.torch import load_file

state_dict = load_file("JEPA_model_ijepa_vith14_1k.safetensors")

print(len(state_dict))
print(list(state_dict.keys()))


JEPA doesn’t rely on a CLS token “summary token” (global representation of the image) It learns representations differently (predictive masking)

In [ ]:
#add extra fake token in jepa model architecture so it matches ViT

import torch

# get JEPA position embeddings
pos_embed = state_dict["embeddings.position_embeddings"]  # (1, 256, 1280)

# create fake CLS token (zeros)
cls_token = torch.zeros(1, 1, pos_embed.shape[-1])

# concatenate → (1, 257, 1280)
new_pos_embed = torch.cat([cls_token, pos_embed], dim=1)

# replace in state_dict
state_dict["embeddings.position_embeddings"] = new_pos_embed

In [ ]:
# loading jepa weights into vit
from transformers import ViTModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# create empty ViT (same structure)
vit_model = ViTModel.from_pretrained("google/vit-huge-patch14-224-in21k")

# load JEPA weights
missing, unexpected = vit_model.load_state_dict(state_dict, strict=False)


print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

vit_model = vit_model.to(device)
vit_model.eval()



In [ ]:
# try forward pass to extract feature
import torch

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    outputs = vit_model(dummy)

tokens = outputs.last_hidden_state  # (1, 257, 1280)

# remove CLS token (first token)
tokens = tokens[:, 1:, :]  # (1, 256, 1280)

# average pooling → final feature vector
features = tokens.mean(dim=1)  # (1, 1280)

print(features.shape)

In [ ]:
# see whic layer is the output of outputs.last_hidden_state
# if true its  583 layernorm
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        activations[name] = output
    return hook

vit_model.layernorm.register_forward_hook(
    hook_fn("final_layernorm")
)
# we can also test if its 582 encoder.layer.31.layernorm_after:  vit_model.encoder.layer[31].layernorm_after.register_forward_hook(


outputs = vit_model(dummy)

torch.allclose(
    outputs.last_hidden_state,
    activations["final_layernorm"]
)


the model gives (B, 257, 1280) but Net2Brain expects (B, 1280)  
Standardizes JEPA to behave like: model(x) → (B, D)

In [ ]:
# varify model(image_batch) → (B, D)
out = jepa_extractor(torch.randn(2,3,224,224).to(device))
print(out.shape)

In [ ]:
# add jepa extractor
fx = FeatureExtractor(
    model=jepa_extractor,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_transformer_extractor
)
fx.extract(
    data_path=stimuli_path,
    save_path='JEPA_Feat_nsd_2'
)

In [1]:
from safetensors.torch import load_file
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_path = "PATH"

from dataclasses import dataclass

@dataclass
class LeJEPAConfig:
    pass
    
ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

state_dict = ckpt["model_state_dict"]


filtered_state_dict = {}

for k, v in state_dict.items():

    # keep ONLY backbone weights
    if k.startswith("backbone."):

        new_key = k.replace("backbone.", "")
        filtered_state_dict[new_key] = v

state_dict = filtered_state_dict

In [5]:

print(ckpt.keys())
print(list(ckpt["model_state_dict"].keys())[:30])

dict_keys(['epoch', 'step', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'history', 'config'])
['backbone.cls_token', 'backbone.pos_embed', 'backbone.patch_embed.proj.weight', 'backbone.patch_embed.proj.bias', 'backbone.blocks.0.norm1.weight', 'backbone.blocks.0.norm1.bias', 'backbone.blocks.0.attn.qkv.weight', 'backbone.blocks.0.attn.qkv.bias', 'backbone.blocks.0.attn.proj.weight', 'backbone.blocks.0.attn.proj.bias', 'backbone.blocks.0.norm2.weight', 'backbone.blocks.0.norm2.bias', 'backbone.blocks.0.mlp.fc1.weight', 'backbone.blocks.0.mlp.fc1.bias', 'backbone.blocks.0.mlp.fc2.weight', 'backbone.blocks.0.mlp.fc2.bias', 'backbone.blocks.1.norm1.weight', 'backbone.blocks.1.norm1.bias', 'backbone.blocks.1.attn.qkv.weight', 'backbone.blocks.1.attn.qkv.bias', 'backbone.blocks.1.attn.proj.weight', 'backbone.blocks.1.attn.proj.bias', 'backbone.blocks.1.norm2.weight', 'backbone.blocks.1.norm2.bias', 'backbone.blocks.1.mlp.fc1.weight', 'backbone.blocks.1.mlp.fc1.bias', '

In [2]:
# get backbone only
filtered_state_dict = {}

for k, v in ckpt["model_state_dict"].items():

    if k.startswith("backbone."):

        new_key = k.replace("backbone.", "")
        filtered_state_dict[new_key] = v

# create timm ViT
import timm

vit_model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
)

#because: you do NOT want classifier head, only backbone embeddings for RSA
filtered_state_dict.pop("pos_embed", None)
filtered_state_dict.pop("head.weight", None)
filtered_state_dict.pop("head.bias", None)


# load LeJEPA backbone weights
missing, unexpected = vit_model.load_state_dict(
    filtered_state_dict,
    strict=False
)

print("Missing:", len(missing))
print("Unexpected:", len(unexpected))

vit_model = vit_model.to(device)
vit_model.eval()

C:\Users\louai\net2brain-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Missing: 1
Unexpected: 0


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [16]:
# testing one img
from PIL import Image

img = Image.open(
    "PATH"
).convert("RGB")

x = my_preprocessor(
    img,
    model_name="lejepa",
    device=device
)
jepa_extractor = JEPAFeatureExtractor(vit_model)
features = my_extactor_lejepa(
    x,
    None,
    jepa_extractor
)

print(features.keys())

for k, v in features.items():
    print(k, v.shape)

dict_keys(['layer_0', 'layer_1', 'layer_2', 'layer_3', 'layer_4', 'layer_5', 'layer_6', 'layer_7', 'layer_8', 'layer_9', 'layer_10', 'layer_11'])
layer_0 torch.Size([1, 768])
layer_1 torch.Size([1, 768])
layer_2 torch.Size([1, 768])
layer_3 torch.Size([1, 768])
layer_4 torch.Size([1, 768])
layer_5 torch.Size([1, 768])
layer_6 torch.Size([1, 768])
layer_7 torch.Size([1, 768])
layer_8 torch.Size([1, 768])
layer_9 torch.Size([1, 768])
layer_10 torch.Size([1, 768])
layer_11 torch.Size([1, 768])


In [7]:
# add jepa extractor

fx = FeatureExtractor(
    model=jepa_extractor,
    device=device,
    preprocessor=my_preprocessor,
    extraction_function=my_extactor_lejepa,
    feature_cleaner=my_cleaner,
)
fx.extract(
    data_path=stimuli_path,
    save_path= "PATH"
    consolidate_per_layer=False
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [07:08<00:00,  2.04it/s, subfiles=1/1]


# LeJepa

In [6]:
#vit-l
import torch

checkpoint_path = "PATH"

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:")
print(ckpt.keys())

print("\nState dict keys:")
print(list(ckpt["state_dict"].keys())[:50])

Checkpoint keys:
dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters', 'wandb', 'spt_run_dir'])

State dict keys:
['callbacks_modules.ordered_queue_embedding.pointer', 'callbacks_modules.ordered_queue_embedding.filled', 'callbacks_modules.ordered_queue_embedding.global_counter', 'callbacks_modules.ordered_queue_embedding.out', 'callbacks_modules.ordered_queue_embedding.order_indices', 'callbacks_modules.ordered_queue_embedding.initialized', 'callbacks_modules.ordered_queue_label.pointer', 'callbacks_modules.ordered_queue_label.filled', 'callbacks_modules.ordered_queue_label.global_counter', 'callbacks_modules.ordered_queue_label.out', 'callbacks_modules.ordered_queue_label.order_indices', 'callbacks_modules.ordered_queue_label.initialized', 'callbacks_modules.linear_probe.0.weight', 'callbacks_modules.linear_probe.0.bias', 'callbacks_modules.linear_probe.1.weight', 'ca

In [13]:
# vit-b
import torch

checkpoint_path = "PATH"

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:")
print(ckpt.keys())

print("\nState dict keys:")
print(list(ckpt["state_dict"].keys())[:50])

Checkpoint keys:
dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters', 'wandb'])

State dict keys:
['callbacks_modules.ordered_queue_embedding.pointer', 'callbacks_modules.ordered_queue_embedding.filled', 'callbacks_modules.ordered_queue_embedding.global_counter', 'callbacks_modules.ordered_queue_embedding.out', 'callbacks_modules.ordered_queue_embedding.order_indices', 'callbacks_modules.ordered_queue_embedding.initialized', 'callbacks_modules.ordered_queue_label.pointer', 'callbacks_modules.ordered_queue_label.filled', 'callbacks_modules.ordered_queue_label.global_counter', 'callbacks_modules.ordered_queue_label.out', 'callbacks_modules.ordered_queue_label.order_indices', 'callbacks_modules.ordered_queue_label.initialized', 'callbacks_modules.linear_probe.0.weight', 'callbacks_modules.linear_probe.0.bias', 'callbacks_modules.linear_probe.1.weight', 'callbacks_modules

In [14]:

filtered_state_dict = {}

for k, v in ckpt["state_dict"].items():
    if k.startswith("model.backbone."):
        new_key = k.replace("model.backbone.", "")
        filtered_state_dict[new_key] = v

print("Number of backbone parameters:", len(filtered_state_dict))
print("First keys:")
print(list(filtered_state_dict.keys())[:20])

Number of backbone parameters: 150
First keys:
['cls_token', 'pos_embed', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc2.weight', 'blocks.0.mlp.fc2.bias', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias']


In [9]:
#vit-l
import timm

vit_model = timm.create_model(
    "vit_large_patch16_224",
    pretrained=False,
    num_classes=0,
)

print("Model created.")

Model created.


In [15]:
# vit-b
import timm

vit_model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
)

print("Model created.")

Model created.


In [16]:
# load the LeJEPA backbone weights into this ViT-L/ViT-B.
missing, unexpected = vit_model.load_state_dict(
    filtered_state_dict,
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

Missing: []
Unexpected: []


### inspecting

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

vit_model = vit_model.to(device)
vit_model.eval()

print("Device:", device)
print("Model ready for feature extraction.")

Device: cuda
Model ready for feature extraction.


In [25]:
#test one image
import torch

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    output = vit_model(dummy)

print("Output shape:", output.shape)

Output shape: torch.Size([1, 768])


### extract feats

In [26]:
jepa_extractor = JEPAFeatureExtractor(vit_model)

#print(jepa_extractor)

In [27]:
# actual layer extractor on one image
from PIL import Image
import os

image_name = os.listdir(stimuli_path)[0]
image_path = os.path.join(stimuli_path, image_name)

image = Image.open(image_path).convert("RGB")

preprocessed = my_preprocessor(
    image,
    "lejepa",
    device
)

features = my_extactor_lejepa(
    preprocessed,
    None,
    jepa_extractor
)

for name, feature in features.items():
    print(name, feature.shape)

layer_0 torch.Size([1, 768])
layer_1 torch.Size([1, 768])
layer_2 torch.Size([1, 768])
layer_3 torch.Size([1, 768])
layer_4 torch.Size([1, 768])
layer_5 torch.Size([1, 768])
layer_6 torch.Size([1, 768])
layer_7 torch.Size([1, 768])
layer_8 torch.Size([1, 768])
layer_9 torch.Size([1, 768])
layer_10 torch.Size([1, 768])
layer_11_pre_norm torch.Size([1, 768])
layer_11_post_norm torch.Size([1, 768])


In [28]:
features = my_extactor_lejepa(
    preprocessed,
    None,
    jepa_extractor
)

for name, feature in features.items():
    print(name, feature.shape)

layer_0 torch.Size([1, 768])
layer_1 torch.Size([1, 768])
layer_2 torch.Size([1, 768])
layer_3 torch.Size([1, 768])
layer_4 torch.Size([1, 768])
layer_5 torch.Size([1, 768])
layer_6 torch.Size([1, 768])
layer_7 torch.Size([1, 768])
layer_8 torch.Size([1, 768])
layer_9 torch.Size([1, 768])
layer_10 torch.Size([1, 768])
layer_11_pre_norm torch.Size([1, 768])
layer_11_post_norm torch.Size([1, 768])


In [29]:
fx = FeatureExtractor(
    model=jepa_extractor,
    device=device,
    preprocessor=my_preprocessor,
    extraction_function=my_extactor_lejepa,
    feature_cleaner=my_cleaner,
)

In [ ]:
fx.extract(
    data_path=stimuli_path,
    save_path="PATH"
    consolidate_per_layer=False
)

Processing files:  28%|█████████████▌                                  | 247/872 [00:11<00:25, 24.36it/s, subfiles=1/1]

#

In [57]:
creator = RDMCreator(verbose=True, device="cpu")

iterator = FeatureIterator(feat_path)

layer, stimuli, feats = next(iter(iterator))

print("Before torch:", feats.shape)

feats = torch.from_numpy(feats).to("cpu")

print("After torch:", feats.shape)

rdm_m = creator._create_rdm(
    feats,
    distance="pearson"
)

print("RDM:", rdm_m.shape)

meta = dict(
    distance="pearson",
    multi_timepoint_rdms=None
)

rdm = LayerRDM(
    rdm=rdm_m,
    layer_name=layer,
    stimuli_name=stimuli,
    meta=meta
)

print("LayerRDM created")

rdm.save(
    "PATH",
    file_format="npz"
)

print("SAVED")

Before torch: (872, 1, 768)
After torch: torch.Size([872, 1, 768])
RDM: torch.Size([872, 872])
LayerRDM created
SAVED


In [58]:
iterator = FeatureIterator(feat_path)

for i, (layer, stimuli, feats) in enumerate(iterator):
    print(i, layer, feats.shape)

0 layer_0 (872, 1, 768)
1 layer_1 (872, 1, 768)
2 layer_2 (872, 1, 768)
3 layer_3 (872, 1, 768)
4 layer_4 (872, 1, 768)
5 layer_5 (872, 1, 768)
6 layer_6 (872, 1, 768)
7 layer_7 (872, 1, 768)
8 layer_8 (872, 1, 768)
9 layer_9 (872, 1, 768)
10 layer_10 (872, 1, 768)
11 layer_11_post_norm (872, 1, 768)
12 layer_11_pre_norm (872, 1, 768)


In [59]:
import os

print(os.listdir(save_path))

['RDM_layer_0.npz', 'RDM_layer_1.npz', 'RDM_layer_10.npz', 'RDM_layer_11_post_norm.npz', 'RDM_layer_11_pre_norm.npz', 'RDM_layer_2.npz', 'RDM_layer_3.npz', 'RDM_layer_4.npz', 'RDM_layer_5.npz', 'RDM_layer_6.npz', 'RDM_layer_7.npz', 'RDM_layer_8.npz', 'RDM_layer_9.npz']
